In [2]:
from typing import TypedDict,List,Literal,Optional

class ApprovalState(TypedDict):
    action_details : str
    status : Optional[Literal["Pending","Approved","Rejected"]]

In [3]:
from langgraph.types import Command, interrupt
#create a node
def approval_node(state: ApprovalState) -> ApprovalState:
    decision = interrupt(
        {
            "question" : "Approve this action?",
            "details" : state["action_details"],
        }
    )

    #Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")

#create a proceed node
def proceed_node(state: ApprovalState):
    return {"status" : "Approved"}

#create a cancle node
def cancel_node(state: ApprovalState):
    return {"status" : "Rejected"}

In [4]:
#Create a state graph

from langgraph.graph import StateGraph,START,END

graph = StateGraph(ApprovalState)

graph.add_node("approval_node",approval_node)
graph.add_node("proceed_node",proceed_node)
graph.add_node("cancel_node",cancel_node)

graph.add_edge(START,"approval_node")
graph.add_edge("proceed_node",END)
graph.add_edge("cancel_node",END)